In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Gold layer tables
fact = spark.table("novacart_dev.gold.fact_order_items")
dim_products = spark.table("novacart_dev.gold.dim_products")
dim_customers = spark.table("novacart_dev.gold.dim_customers")
dim_dates = spark.table("novacart_dev.gold.dim_dates")
dim_orders = spark.table("novacart_dev.gold.dim_orders")

print("✓ Gold layer tables loaded successfully")
print(f"  - Fact table: {fact.count():,} line items")
print(f"  - Products: {dim_products.count():,}")
print(f"  - Customers: {dim_customers.count():,}")
print(f"  - Orders: {dim_orders.count():,}")

In [0]:
# KPI 1: Total Revenue
total_revenue = fact.agg(
    F.sum("revenue_usd").alias("total_revenue_usd")
).collect()[0]

print("="*50)
print("KPI 1: TOTAL REVENUE")
print("="*50)
print(f"Total Revenue: ${total_revenue['total_revenue_usd']:,.2f}")
print("="*50)

In [0]:
# KPI 2: Revenue by Country
revenue_by_country = fact.groupBy("order_country") \
    .agg(
        F.sum("revenue_usd").alias("revenue_usd"),
        F.count("order_item_key").alias("order_items")
    ) \
    .orderBy(F.desc("revenue_usd"))

print("="*50)
print("KPI 2: REVENUE BY COUNTRY")
print("="*50)
display(revenue_by_country)

In [0]:
# KPI 3: Revenue by Channel
revenue_by_channel = fact.groupBy("channel") \
    .agg(
        F.sum("revenue_usd").alias("revenue_usd"),
        F.count("order_item_key").alias("order_items"),
        F.countDistinct("order_key").alias("orders")
    ) \
    .orderBy(F.desc("revenue_usd"))

print("="*50)
print("KPI 3: REVENUE BY CHANNEL")
print("="*50)
display(revenue_by_channel)

In [0]:
# KPI 4: Completed Order Count
completed_orders = fact.filter(F.col("order_status_clean") == "completed") \
    .select("order_key") \
    .distinct() \
    .count()

print("="*50)
print("KPI 4: COMPLETED ORDER COUNT")
print("="*50)
print(f"Completed Orders: {completed_orders:,}")
print("="*50)

In [0]:
# KPI 5: Completed Order Rate
total_orders = fact.select("order_key").distinct().count()
completed_order_rate = (completed_orders / total_orders * 100) if total_orders > 0 else 0

print("="*50)
print("KPI 5: COMPLETED ORDER RATE")
print("="*50)
print(f"Total Orders: {total_orders:,}")
print(f"Completed Orders: {completed_orders:,}")
print(f"Completion Rate: {completed_order_rate:.2f}%")
print("="*50)

In [0]:
# KPI 6: Average Order Value (AOV)
aov = fact.groupBy("order_key") \
    .agg(F.sum("revenue_usd").alias("order_total")) \
    .agg(
        F.avg("order_total").alias("avg_order_value"),
        F.min("order_total").alias("min_order_value"),
        F.max("order_total").alias("max_order_value")
    ).collect()[0]

print("="*50)
print("KPI 6: AVERAGE ORDER VALUE (AOV)")
print("="*50)
print(f"Average Order Value: ${aov['avg_order_value']:,.2f}")
print(f"Min Order Value: ${aov['min_order_value']:,.2f}")
print(f"Max Order Value: ${aov['max_order_value']:,.2f}")
print("="*50)

In [0]:
# KPI 7: Top 5 Products by Revenue
# Filter out null product keys (orphan products)
top_products = fact.filter(F.col("product_key").isNotNull()) \
    .join(dim_products, fact.product_key == dim_products.product_key, "left") \
    .groupBy(fact.product_key, dim_products.product_name, dim_products.category) \
    .agg(
        F.sum("revenue_usd").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.countDistinct(fact.order_key).alias("order_count")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .limit(5)

print("="*50)
print("KPI 7: TOP 5 PRODUCTS BY REVENUE")
print("="*50)
display(top_products)

In [0]:
# KPI 8: Active Customers Count
# Customers who have placed at least one completed order
active_customers = fact.filter(F.col("order_status_clean") == "completed") \
    .select("customer_key") \
    .distinct() \
    .count()

total_customers = dim_customers.count()
active_rate = (active_customers / total_customers * 100) if total_customers > 0 else 0

print("="*50)
print("KPI 8: ACTIVE CUSTOMERS COUNT")
print("="*50)
print(f"Active Customers: {active_customers:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Active Rate: {active_rate:.2f}%")
print("="*50)

In [0]:
# KPI 9: Customer Acquisition by Month
# Filter out customers with null registration dates
customer_acquisition = dim_customers \
    .filter(
        F.col("registration_year").isNotNull() & 
        F.col("registration_month").isNotNull()
    ) \
    .groupBy("registration_year", "registration_month") \
    .agg(
        F.count("customer_key").alias("new_customers")
    ) \
    .withColumn(
        "year_month",
        F.concat(
            F.col("registration_year").cast("string"),
            F.lit("-"),
            F.lpad(F.col("registration_month").cast("string"), 2, "0")
        )
    ) \
    .orderBy("registration_year", "registration_month")

print("="*50)
print("KPI 9: CUSTOMER ACQUISITION BY MONTH")
print("="*50)
display(customer_acquisition)

In [0]:
# KPI 10: Data Quality Score
# Calculate percentage of records without data quality issues
total_records = fact.count()

# Count records with any DQ flag set to 1
records_with_issues = fact.filter(
    (F.col("dq_line_total_mismatch") == 1) |
    (F.col("dq_orphan_product") == 1) |
    (F.col("dq_missing_order_item_id") == 1) |
    (F.col("dq_invalid_quantity") == 1) |
    (F.col("dq_invalid_price") == 1)
).count()

clean_records = total_records - records_with_issues
data_quality_score = (clean_records / total_records * 100) if total_records > 0 else 0

# Breakdown by issue type
issue_breakdown = fact.select(
    F.sum("dq_line_total_mismatch").alias("line_total_mismatch"),
    F.sum("dq_orphan_product").alias("orphan_product"),
    F.sum("dq_missing_order_item_id").alias("missing_order_item_id"),
    F.sum("dq_invalid_quantity").alias("invalid_quantity"),
    F.sum("dq_invalid_price").alias("invalid_price")
).collect()[0]

print("="*50)
print("KPI 10: DATA QUALITY SCORE")
print("="*50)
print(f"Total Records: {total_records:,}")
print(f"Clean Records: {clean_records:,}")
print(f"Records with Issues: {records_with_issues:,}")
print(f"\nData Quality Score: {data_quality_score:.2f}%")
print("\nIssue Breakdown:")
print(f"  - Line Total Mismatch: {issue_breakdown['line_total_mismatch']:,}")
print(f"  - Orphan Product: {issue_breakdown['orphan_product']:,}")
print(f"  - Missing Order Item ID: {issue_breakdown['missing_order_item_id']:,}")
print(f"  - Invalid Quantity: {issue_breakdown['invalid_quantity']:,}")
print(f"  - Invalid Price: {issue_breakdown['invalid_price']:,}")
print("="*50)

In [0]:
# KPI Summary Dashboard
print("="*70)
print("               NOVOCART GOLD LAYER - KPI SUMMARY")
print("="*70)
print(f"\n1. Total Revenue:              ${total_revenue['total_revenue_usd']:,.2f}")
print(f"2. Completed Orders:           {completed_orders:,}")
print(f"3. Completed Order Rate:       {completed_order_rate:.2f}%")
print(f"4. Average Order Value:        ${aov['avg_order_value']:,.2f}")
print(f"5. Active Customers:           {active_customers:,} / {total_customers:,} ({active_rate:.2f}%)")
print(f"6. Data Quality Score:         {data_quality_score:.2f}%")
print(f"\n7. Top Revenue Country:        [See KPI 2 for details]")
print(f"8. Top Revenue Channel:        [See KPI 3 for details]")
print(f"9. Top 5 Products:             [See KPI 7 for details]")
print(f"10. Customer Acquisition:      [See KPI 9 for monthly trends]")
print("="*70)
print(f"\nReport Generated: {F.current_timestamp()}")
print("="*70)